In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)


def _parse_percent_series(s: pd.Series | None, *, index: pd.Index) -> pd.Series:
    """Parse numbers or strings like '6.81%' into floats."""
    if s is None:
        return pd.Series(np.nan, index=index, dtype="float64")

    return pd.to_numeric(
        s.astype(str).str.replace("%", "", regex=False).str.strip(),
        errors="coerce",
    )


def _first_non_null_numeric(*cols: pd.Series | None, index: pd.Index | None = None) -> pd.Series:
    """Take the first available numeric value across candidate columns."""
    if index is None:
        first_existing = next((c for c in cols if c is not None), None)
        if first_existing is None:
            raise ValueError("No columns were provided and index is missing")
        index = first_existing.index

    out = pd.Series(np.nan, index=index, dtype="float64")
    for c in cols:
        if c is None:
            continue
        out = out.combine_first(pd.to_numeric(c, errors="coerce"))
    return out


@dataclass(frozen=True)
class EtfSource:
    etf: str
    file: str
    loader: Callable[[pd.DataFrame, str], pd.DataFrame]


def load_ishares_like(df: pd.DataFrame, etf: str) -> pd.DataFrame:
    out = pd.DataFrame(
        {
            "ETF": etf,
            "Ticker": df.get("Ticker"),
            "percent": _first_non_null_numeric(
                df.get("Weight (%)"),
                df.get("Market Weight"),
                index=df.index,
            ),
            "Name": df.get("Name"),
            "Sector": df.get("Sector"),
            "Location": df.get("Location"),
        }
    )
    return out


def load_vanguard_or_smh(df: pd.DataFrame, etf: str) -> pd.DataFrame:
    a = _parse_percent_series(df.get("% of market value"), index=df.index)
    b = _parse_percent_series(df.get("% of Net Assets"), index=df.index)

    name_left = df.get("Holding name")
    if name_left is None:
        name_left = pd.Series(np.nan, index=df.index, dtype="object")

    name_right = df.get("Holding Name")
    if name_right is None:
        name_right = pd.Series(np.nan, index=df.index, dtype="object")

    out = pd.DataFrame(
        {
            "ETF": etf,
            "Ticker": df.get("Ticker"),
            "percent": _first_non_null_numeric(a, b, index=df.index),
            "Name": name_left.combine_first(name_right),
            "Sector": df.get("Sector"),
            "Region": df.get("Region"),
        }
    )
    return out


def load_fidelity_like(df: pd.DataFrame, etf: str) -> pd.DataFrame:
    out = pd.DataFrame(
        {
            "ETF": etf,
            "Ticker": df.get("Symbol"),
            "percent": pd.to_numeric(df.get("Weight"), errors="coerce"),
            "Name": df.get("Company"),
        }
    )
    return out


def load_all_holdings(data_dir: Path, sources: list[EtfSource]) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []

    for src in sources:
        raw = pd.read_csv(data_dir / src.file, keep_default_na=False, na_values=[""])
        std = src.loader(raw, src.etf)

        std["ETF"] = std["ETF"].astype(str)
        std["Ticker"] = std["Ticker"].astype(str).str.strip()
        std.loc[std["Ticker"].isin(["", "nan", "None"]), "Ticker"] = np.nan

        std = std.dropna(subset=["ETF", "Ticker", "percent"])
        std = std[std["percent"].between(0, 100, inclusive="both")]

        frames.append(std)

    return pd.concat(frames, ignore_index=True)


def compute_metrics(holdings: pd.DataFrame, positions_usd_by_etf: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    df = holdings.copy()

    df["my_positions"] = df["ETF"].map(positions_usd_by_etf).fillna(0.0)
    df["my_values"] = df["my_positions"] * df["percent"] / 100.0

    total_invested = float(df["my_values"].sum())
    df["concentration"] = 0.0 if total_invested <= 0 else df["my_values"] / total_invested

    concentration_by_ticker = (
        df.groupby("Ticker", dropna=False)["concentration"].sum().sort_values(ascending=False)
    )

    return df, concentration_by_ticker


In [2]:
DATA_DIR = Path(".")

sources = [
    # iShares-style
    *[
        EtfSource(etf, f"{etf}.csv", load_ishares_like)
        for etf in ["AQLT", "IEFA", "IJH", "IVV", "IWY", "JPXN", "EFAV", "QUAL"]
    ],
    # Vanguard + SMH
    EtfSource("VCN", "VCN.csv", load_vanguard_or_smh),
    EtfSource("SMH", "SMH.csv", load_vanguard_or_smh),
    # Fidelity
    EtfSource("FTEC", "FTEC.csv", load_fidelity_like),
    EtfSource("FQAL", "FQAL.csv", load_fidelity_like),
]

# Edit these values (or load them from a CSV) to match your portfolio.
positions = pd.Series(
    {
        "AQLT": 100,
        "IEFA": 100,
        "IJH": 100,
        "IVV": 100,
        "IWY": 100,
        "JPXN": 100,
        "EFAV": 100,
        "QUAL": 100,
        "VCN": 100,
        "SMH": 100,
        "FTEC": 100,
        "FQAL": 100,
    },
    dtype="float64",
)

holdings = load_all_holdings(DATA_DIR, sources)
holdings.head()

,ETF,Ticker,percent,Name,Sector,Location,Region
0,AQLT,2330,5.88,TAIWAN SEMICONDUCTOR MANUFACTURING,Information Technology,Taiwan,NaN
1,AQLT,META,5.13,META PLATFORMS INC CLASS A,Communication,United States,NaN
2,AQLT,NVDA,4.86,NVIDIA CORP,Information Technology,United States,NaN
3,AQLT,AAPL,4.66,APPLE INC,Information Technology,United States,NaN
4,AQLT,MSFT,3.81,MICROSOFT CORP,Information Technology,United States,NaN


In [3]:
df_combined, concentration_by_ticker = compute_metrics(holdings, positions)

# Export full combined holdings + your look-through metrics
out_path = Path("my_data.csv")
df_combined.to_csv(out_path, index=False)

print(f"Total invested: ${df_combined['my_values'].sum():,.2f}")
concentration_by_ticker.head(20)

Total invested: $1,197.67


Ticker
NVDA     0.066237
AAPL     0.044227
MSFT     0.032964
AVGO     0.019830
GOOGL    0.014754
META     0.014344
LRCX     0.011472
KLAC     0.010011
AMAT     0.009560
LLY      0.008926
TSM      0.008850
GOOG     0.008583
V        0.007965
ASML     0.007849
AMZN     0.007297
AMD      0.007272
MU       0.006237
QCOM     0.006028
MA       0.006020
TXN      0.005786
Name: concentration, dtype: float64

In [4]:
def compare_etfs_top_holdings(
    holdings: pd.DataFrame,
    etfs: list[str],
    top_n: int = 30,
) -> pd.DataFrame:
    """Compare top holdings side by side using each ETF's own percent weights."""
    df = holdings[holdings["ETF"].isin(etfs)].copy()

    # Some files can contain duplicate ticker rows; aggregate first.
    per_etf = (
        df.groupby(["ETF", "Ticker"], as_index=False)["percent"]
        .sum()
        .rename(columns={"percent": "weight_pct"})
    )

    top = (
        per_etf.sort_values(["ETF", "weight_pct"], ascending=[True, False])
        .groupby("ETF", group_keys=False)
        .head(top_n)
    )

    comparison = (
        top.pivot(index="Ticker", columns="ETF", values="weight_pct")
        .fillna(0.0)
    )

    # Order rows by total weight across selected ETFs to surface important names first.
    comparison = comparison.loc[comparison.sum(axis=1).sort_values(ascending=False).index]

    return comparison


etfs_to_compare = ["AQLT", "FQAL", "QUAL","IWY"]

# Make this cell robust if run independently.
if "holdings" not in globals():
    holdings = load_all_holdings(DATA_DIR, sources)

missing_etfs = sorted(set(etfs_to_compare) - set(holdings["ETF"].dropna().unique()))
if missing_etfs:
    raise ValueError(f"These ETFs were not found in loaded data: {missing_etfs}")

top_30_side_by_side = compare_etfs_top_holdings(holdings, etfs_to_compare, top_n=30)
top_30_side_by_side.head(30)


ETF,AQLT,FQAL,IWY,QUAL
Ticker,,,,
NVDA,4.86,7.56,15.21,6.61
AAPL,4.66,6.71,12.68,6.33
MSFT,3.81,5.06,10.05,5.18
META,5.13,2.21,3.60,3.84
GOOGL,2.97,5.10,4.15,2.23
LLY,2.43,1.60,2.59,2.88
V,2.34,1.38,2.02,2.94
AVGO,0.00,3.02,5.00,0.00
GOOG,2.49,0.00,3.36,1.86


In [5]:

etfs_to_compare = ["IWY", "IVV", "FTEC"]

# Make this cell robust if run independently.
if "holdings" not in globals():
    holdings = load_all_holdings(DATA_DIR, sources)

missing_etfs = sorted(set(etfs_to_compare) - set(holdings["ETF"].dropna().unique()))
if missing_etfs:
    raise ValueError(f"These ETFs were not found in loaded data: {missing_etfs}")

top_30_side_by_side = compare_etfs_top_holdings(holdings, etfs_to_compare, top_n=30)
top_30_side_by_side.head(30)

ETF,FTEC,IVV,IWY
Ticker,,,
NVDA,18.25,8.04,15.21
AAPL,16.02,6.57,12.68
MSFT,10.29,5.09,10.05
AVGO,4.32,3.10,5.00
AMZN,0.00,3.97,4.77
GOOGL,0.00,3.22,4.15
META,0.00,2.40,3.60
GOOG,0.00,2.57,3.36
TSLA,0.00,1.81,3.34


In [6]:
etfs_to_compare = ["IVV", "IJH"]

# Make this cell robust if run independently.
if "holdings" not in globals():
    holdings = load_all_holdings(DATA_DIR, sources)

missing_etfs = sorted(set(etfs_to_compare) - set(holdings["ETF"].dropna().unique()))
if missing_etfs:
    raise ValueError(f"These ETFs were not found in loaded data: {missing_etfs}")

top_30_side_by_side = compare_etfs_top_holdings(holdings, etfs_to_compare, top_n=30)
top_30_side_by_side.head(30)

ETF,IJH,IVV
Ticker,,
NVDA,0.00,8.04
AAPL,0.00,6.57
MSFT,0.00,5.09
AMZN,0.00,3.97
GOOGL,0.00,3.22
AVGO,0.00,3.10
GOOG,0.00,2.57
META,0.00,2.40
TSLA,0.00,1.81
